# V5W_07 - Geometria di Riemann: asse fenotipico + decoding baseline

Le matrici di connettivita/covarianza sono SPD -> varieta di Riemann. Si usa la geometria corretta (covarianze + tangent space) per:
1. **Asse fenotipico**: media di Riemann per soggetto -> tangent space -> PCA. Se i soggetti si **allineano** su PC1, il continuum e una geodetica. Correlazione Tangent-PC1 vs PI euclideo vs alpha mu.
2. **Decoding baseline**: MDM + TangentSpace+LR (gold-standard EEG classico), subject-specific LOSO. Se chance -> ceiling blindato.

Covarianza per trial: estimatore OAS (SPD ben condizionata). **Env: `daniele_311`** (pyriemann sul server). Richiede cache V5W_05 (PI) e V5W_06 (alpha, opzionale).

## par.1 - Config + PI/alpha dalle cache

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from tqdm.auto import tqdm
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace
from pyriemann.utils.mean import mean_riemann
from pyriemann.classification import MDM

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('v5w07')
project_root = next((p for p in [Path.cwd()]+list(Path.cwd().parents) if (p/'.git').exists()), Path.cwd())
FIG_DIR  = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
V5W05    = project_root / 'models' / 'v5w05'
V5W06    = project_root / 'models' / 'v5w06'
CKPT_DIR = project_root / 'models' / 'v5w07'; CKPT_DIR.mkdir(parents=True, exist_ok=True)
CSV_ROOT = project_root / 'data' / '5words_subjects'
N_CHAN, FS, N_CLASSES = 61, 256, 5
triu_idx = np.triu_indices(N_CHAN, k=1)
word2label = json.loads((project_root/'configs'/'label_schemes'/'label2idx_5words.json').read_text())
_PAT = re.compile(r'^P(\d+)_S(\d+)$')

_feat = np.load(V5W05/'feat_abs_pcc.npz', allow_pickle=True)
FEAT_G, SUBJ_F = _feat['feat_g'], _feat['subj'].tolist()
_lab = np.load(V5W05/'cluster_labels.npz', allow_pickle=True)
lab_map = {int(s): int(l) for s, l in zip(_lab['subj'], _lab['labels'])}
def vec_to_sym(v):
    M = np.zeros((N_CHAN, N_CHAN)); M[triu_idx] = v; return M + M.T
NS = np.array([vec_to_sym(FEAT_G[i]).sum(1)/(N_CHAN-1) for i in range(len(SUBJ_F))])
lab_arr = np.array([lab_map[s] for s in SUBJ_F])
_diff = NS[lab_arr==1].mean(0) - NS[lab_arr==0].mean(0); _diff /= (np.linalg.norm(_diff)+1e-12)
PI = {s: float(p) for s, p in zip(SUBJ_F, NS @ _diff)}
ALPHA = {}
if (V5W06/'subject_alpha.npz').exists():
    z = np.load(V5W06/'subject_alpha.npz', allow_pickle=True)
    ALPHA = {int(s): float(a) for s, a in zip(z['subj'], z['alpha'])}
log.info(f'PI: {len(PI)} sogg.  alpha: {len(ALPHA)} sogg.')


## par.2 - Covarianze per-trial (SPD)

In [ ]:
CACHE = CKPT_DIR / 'covs.npz'
if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    COVS, SUBJ, SESS, LAB = z['covs'], z['subj'], z['sess'], z['lab']
    log.info(f'Covarianze da cache: {COVS.shape}')
else:
    cov_est = Covariances(estimator='oas')
    covs_all, subj_all, sess_all, lab_all = [], [], [], []
    sdirs = sorted(d for d in CSV_ROOT.iterdir() if _PAT.match(d.name))
    for sd in tqdm(sdirs, desc='covarianze'):
        m = _PAT.match(sd.name); sid, ses = int(m.group(1)), int(m.group(2))
        X, y = [], []
        for csv in sorted(sd.glob('*_img_*.csv')):
            if csv.name.startswith('._'): continue
            w = csv.name.split('_img_')[0]
            if w not in word2label: continue
            arr = pd.read_csv(csv, header=None).values.astype(np.float32)
            if arr.shape != (N_CHAN, 384): continue
            X.append(arr); y.append(word2label[w])
        if not X: continue
        C = cov_est.transform(np.stack(X))
        covs_all.append(C.astype(np.float32))
        subj_all += [sid]*len(y); sess_all += [ses]*len(y); lab_all += y
    COVS = np.concatenate(covs_all); SUBJ = np.array(subj_all); SESS = np.array(sess_all); LAB = np.array(lab_all)
    np.savez(CACHE, covs=COVS, subj=SUBJ, sess=SESS, lab=LAB)
    log.info(f'Covarianze calcolate e salvate: {COVS.shape}')
ALL_SUBJ = sorted(set(SUBJ.tolist()))
print(f'Trial: {len(COVS)}  soggetti: {len(ALL_SUBJ)}')


## par.3 - Asse fenotipico Riemann (allineamento)

In [ ]:
subj_mean = []
for sid in tqdm(ALL_SUBJ, desc='media Riemann/sogg'):
    subj_mean.append(mean_riemann(COVS[SUBJ == sid]))
M = np.stack(subj_mean)
ts = TangentSpace().fit(M)
TS = ts.transform(M)
Z = PCA(n_components=5, random_state=42).fit_transform(StandardScaler().fit_transform(TS))

pi_arr = np.array([PI.get(s, np.nan) for s in ALL_SUBJ])
if np.corrcoef(Z[:, 0], np.nan_to_num(pi_arr))[0, 1] < 0:
    Z[:, 0] = -Z[:, 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sc = axes[0].scatter(Z[:, 0], Z[:, 1], c=pi_arr, cmap='RdBu_r', s=90, edgecolor='k', lw=0.5)
for s, zx, zy in zip(ALL_SUBJ, Z[:, 0], Z[:, 1]):
    axes[0].annotate(f'P{s}', (zx, zy), fontsize=6, alpha=0.6)
axes[0].set_xlabel('Tangent PC1'); axes[0].set_ylabel('Tangent PC2')
axes[0].set_title('Soggetti nel tangent space (Riemann), colore = PI', fontweight='bold')
plt.colorbar(sc, ax=axes[0], label='PI')

m = ~np.isnan(pi_arr)
r_pi, p_pi = pearsonr(Z[m, 0], pi_arr[m])
axes[1].scatter(pi_arr[m], Z[m, 0], s=70, edgecolor='k', lw=0.5, color='#444')
axes[1].set_xlabel('PI euclideo (triu)'); axes[1].set_ylabel('Tangent PC1 (Riemann)')
axes[1].set_title(f'Riemann vs Euclideo: r={r_pi:.3f} (p={p_pi:.1e})', fontweight='bold')
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w07_riemann_axis.png', dpi=160, bbox_inches='tight'); plt.show()

print('='*60)
print(f'  Tangent-PC1 vs PI euclideo:  r={r_pi:+.3f}  p={p_pi:.2e}')
if ALPHA:
    al = np.array([ALPHA.get(s, np.nan) for s in ALL_SUBJ]); ma = ~np.isnan(al)
    r_a, p_a = pearsonr(Z[ma, 0], al[ma])
    print(f'  Tangent-PC1 vs alpha mu:     r={r_a:+.3f}  p={p_a:.2e}')
print('  -> PC1 allineato con PI e alpha => asse confermato in 3 modi indipendenti')
print('='*60)
np.savez(CKPT_DIR/'tangent_axis.npz', subj=np.array(ALL_SUBJ), pc1=Z[:, 0], pc=Z)


## par.4 - Decoding baseline Riemann

In [ ]:
rows = []
for sid in tqdm(ALL_SUBJ, desc='Riemann decoding'):
    msk = SUBJ == sid
    covs_s, lab_s, sess_s = COVS[msk], LAB[msk], SESS[msk]
    us = sorted(set(sess_s.tolist()))
    if len(us) < 2:
        continue
    te = us[-1]
    tr_m, te_m = sess_s != te, sess_s == te
    if tr_m.sum() == 0 or te_m.sum() == 0:
        continue
    Xtr, ytr, Xte, yte = covs_s[tr_m], lab_s[tr_m], covs_s[te_m], lab_s[te_m]
    try:
        b_mdm = balanced_accuracy_score(yte, MDM().fit(Xtr, ytr).predict(Xte))
    except Exception:
        b_mdm = np.nan
    try:
        tslr = make_pipeline(TangentSpace(), LogisticRegression(max_iter=1000, C=1.0))
        tslr.fit(Xtr, ytr)
        b_ts = balanced_accuracy_score(yte, tslr.predict(Xte))
    except Exception:
        b_ts = np.nan
    rows.append((sid, b_mdm, b_ts))

df = pd.DataFrame(rows, columns=['subj', 'MDM', 'TS_LR'])
chance = 1 / N_CLASSES
import torch
dh = project_root/'models'/'v5w03'
b_dh = np.array([float(torch.load(c, weights_only=False)['test_bacc']) for c in sorted(dh.glob('P*.pt'))]) if dh.exists() else np.array([])

print('='*60)
print(f'  V5W_07 - Decoding Riemann subject-specific ({len(df)} sogg., chance {chance:.0%})')
print('='*60)
for col in ['MDM', 'TS_LR']:
    v = df[col].dropna().values
    print(f'  {col:6s}: mean={v.mean():.4f}  median={np.median(v):.4f}  max={v.max():.4f}  >chance={(v>chance).mean()*100:.0f}%')
if len(b_dh):
    print(f'  DHSLP : mean={b_dh.mean():.4f}  (V5W_03, riferimento)')
print('  -> se anche MDM/TS-LR (gold-standard EEG classico) sono chance, ceiling blindato')
print('='*60)

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(np.arange(len(df))-0.2, df['MDM'], 0.4, label='MDM', color='#1f77b4', alpha=0.8)
ax.bar(np.arange(len(df))+0.2, df['TS_LR'], 0.4, label='TS+LR', color='#ff7f0e', alpha=0.8)
ax.axhline(chance, color='k', ls='--', lw=1.5, label=f'Chance ({chance:.0%})')
ax.set_xticks(range(len(df))); ax.set_xticklabels([f'P{s}' for s in df['subj']], rotation=90, fontsize=6)
ax.set_ylabel('Balanced Accuracy'); ax.set_title('V5W_07 - Riemann decoding (5 parole)'); ax.legend()
plt.tight_layout(); plt.savefig(FIG_DIR/'v5w07_riemann_decoding.png', dpi=150, bbox_inches='tight'); plt.show()
df.to_csv(FIG_DIR/'v5w07_riemann_decoding.csv', index=False)
